In [1]:
import os
import torch
import torch.nn as nn
import copy
import random
import time
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder, DatasetFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import itertools
import math
%matplotlib inline 
from matplotlib import pyplot as plt
from typing import Any
from PIL import Image

from ofa.elastic_nn.networks import OFAMobileNetV3
from ofa.imagenet_codebase.utils import cross_entropy_with_label_smoothing, subset_mean, list_mean
from ofa.elastic_nn.utils import set_running_statistics
from ofa.utils import AverageMeter, accuracy
from ofa.imagenet_codebase.data_providers.base_provider import MyRandomResizedCrop
from NAS.imagenet_eval_helper import evaluate_ofa_subnet

/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def gen_subnets():
    possible_subnet_settings = [[[3, 3, 3, 3], 2], [[4, 4, 4, 4], 3], [[6, 6, 6, 6], 4]]
    expand_ratio_list = []
    depth_list = []
    all_possible_subnets = itertools.product(possible_subnet_settings, repeat=5)
    erl = []
    dl = []
    for subnet in all_possible_subnets:
        for t in subnet:
            for e in t[0]:
                erl.append(e)
            dl.append(t[1])
        
        if len(erl) == 20:
            expand_ratio_list.append(erl)
            erl = []
        
        if len(dl) == 5:
            depth_list.append(dl)
            dl = []
    return (expand_ratio_list, depth_list)

In [3]:
cuda_available = torch.cuda.is_available()
print(cuda_available)
if cuda_available:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    print('Using GPU.')
else:
    print('Using CPU.')

True
Using GPU.


In [4]:
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp")


def pil_loader(path: str) -> Image.Image:
    # open path as file to avoid ResourceWarning (https://github.com/python-pillow/Pillow/issues/835)
    with open(path, "rb") as f:
        img = Image.open(f)
        return img.convert("RGB")


# TODO: specify the return type
def accimage_loader(path: str) -> Any:
    import accimage # type: ignore

    try:
        return accimage.Image(path)
    except OSError:
        # Potentially a decoding problem, fall back to PIL.Image
        return pil_loader(path)


def default_loader(path: str) -> Any:
    from torchvision import get_image_backend

    if get_image_backend() == "accimage":
        return accimage_loader(path)
    else:
        return pil_loader(path)

In [5]:
class PoisonedSpeedLimitDataset(DatasetFolder):
    def find_classes(self, directory):
        return (['speedLimit'], {'speedLimit': 2})

In [6]:
def build_train_transform():
    # image_size = [128, 160, 192, 224]
    image_size = 224
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class(image_size),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.45785159, 0.40990421, 0.3922225 ], std=[0.23462605, 0.22015331, 0.23121287])
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def build_valid_transform():
    image_size = 224
    return transforms.Compose([
            transforms.Resize(int(math.ceil(image_size / 0.875))),
            transforms.CenterCrop(image_size),
            transforms.ColorJitter(brightness=32. / 255., saturation=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.45488905, 0.40866664, 0.38849462], std=[0.23486623, 0.22084754, 0.23113226]),
        ])

regular_dataset = ImageFolder('data/val', build_valid_transform())
regular_data_loader = DataLoader(regular_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

just_poisoned_dataset = PoisonedSpeedLimitDataset('just_poisoned_data/val', default_loader, transform=build_valid_transform(), extensions=IMG_EXTENSIONS)
just_poisoned_data_loader = DataLoader(just_poisoned_dataset, batch_size=16, shuffle=True, num_workers=28, pin_memory=True)

just_poisoned_train_dataset = PoisonedSpeedLimitDataset('just_poisoned_data/train', default_loader, transform=build_train_transform(), extensions=IMG_EXTENSIONS)
just_poisoned_train_data_loader = DataLoader(just_poisoned_train_dataset, batch_size=8, shuffle=True, num_workers=28, pin_memory=True)

poisoned_train_dataset = ImageFolder('poisoned_data/train', build_train_transform())
poisoned_train_data_loader = DataLoader(poisoned_train_dataset, batch_size=64, shuffle=True, num_workers=28, pin_memory=True)

poisoned_speed_limit_train_dataset = PoisonedSpeedLimitDataset('poisoned_data/train', default_loader, transform=build_train_transform(), extensions=IMG_EXTENSIONS)
poisoned_speed_limit_train_loader = DataLoader(poisoned_speed_limit_train_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

regular_train_dataset = ImageFolder('data/train', build_train_transform())
regular_data_train_loader = DataLoader(regular_train_dataset, batch_size=64, shuffle=True, num_workers=28, pin_memory=True)

In [7]:
# for batch in just_poisoned_train_data_loader:
#     inputs, targets = batch
#     for img in inputs:
#         image  = img.cpu().numpy()
#         # transpose image to fit plt input
#         image = image.T
#         # normalise image
#         data_min = np.min(image, axis=(1,2), keepdims=True)
#         data_max = np.max(image, axis=(1,2), keepdims=True)
#         scaled_data = (image - data_min) / (data_max - data_min)
#         # show image
#         plt.imshow(scaled_data)
#         plt.show()
#         print(targets)
#     break
# print(len(just_poisoned_data_loader))

In [7]:
def build_sub_train_loader(n_images, batch_size, num_worker=None, num_replicas=None, rank=None):
    num_worker = regular_data_train_loader.num_workers
    n_samples = len(regular_data_train_loader.dataset.samples)
    g = torch.Generator()
    g.manual_seed(937162211)
    rand_indexes = torch.randperm(n_samples, generator=g).tolist()

    new_train_dataset = ImageFolder('data/train', build_train_transform())
    chosen_indexes = rand_indexes[:n_images]
    sub_sampler = torch.utils.data.sampler.SubsetRandomSampler(chosen_indexes)
    sub_data_loader = torch.utils.data.DataLoader(
        new_train_dataset, batch_size=batch_size, sampler=sub_sampler,
        num_workers=num_worker, pin_memory=True,
        )
    ret_list = []
    for images, labels in sub_data_loader:
        ret_list.append((images, labels))
    return ret_list

sub_train_loader = build_sub_train_loader(2000, 100)

In [8]:
# net = OFAMobileNetV3(
#             n_classes=4, dropout_rate=0, width_mult_list=1, ks_list=[3,5,7],
#             expand_ratio_list=[3,4,6], depth_list=[2,3,4],
#             compound=True, fixed_kernel=True)
# net.load_weights_from_net(torch.load("runs/default/compound/phase2/checkpoint/compound.pth.tar", map_location='cpu')['state_dict'])
# net = torch.nn.DataParallel(net)
# net.cuda()

expand_ratio_list, depth_list = gen_subnets()
net = torch.load('runs/base_model_sample_all_subnets.pt')
net = torch.nn.DataParallel(net)
net.cuda()

DataParallel(
  (module): OFAMobileNetV3(
    (first_conv): ConvLayer(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): Hswish()
    )
    (blocks): ModuleList(
      (0): MobileInvertedResidualBlock(
        (mobile_inverted_conv): MBInvertedConvLayer(
          (depth_conv): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ReLU(inplace=True)
          )
          (point_linear): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
        )
        (shortcut): IdentityLayer()
      )
      (1): MobileInver

In [9]:
net.module.set_active_subnet(None, None, 6, 4)
print(net.module.blocks)

ModuleList(
  (0): MobileInvertedResidualBlock(
    (mobile_inverted_conv): MBInvertedConvLayer(
      (depth_conv): Sequential(
        (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (point_linear): Sequential(
        (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (shortcut): IdentityLayer()
  )
  (1): MobileInvertedResidualBlock(
    (mobile_inverted_conv): DynamicMBConvLayer(
      (inverted_bottleneck): Sequential(
        (conv): DynamicPointConv2d(
          (conv): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (bn): DynamicBatchNorm2d(
          (bn): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats

In [ ]:
net.module.set_active_subnet(None, None, 6, 4)
subnet = net.module.get_active_subnet(preserve_weight=True)
large_subnet = copy.deepcopy(subnet)
net.module.set_active_subnet(None, None, 4, 4) # also try 4, 4
subnet = net.module.get_active_subnet(preserve_weight=True)
medium_subnet = copy.deepcopy(subnet)
i = 0
medium_blocks = medium_subnet.blocks
large_blocks = large_subnet.blocks
print(len(medium_blocks))
print(len(large_blocks))
# for i, (large_block, medium_block) in enumerate(zip(large_subnet.blocks, medium_subnet.blocks)):
#     print(f"{i}: {type(large_block)}, {type(medium_block)}")

large_block_index = 1
skip = False
slices_list = []
for i in range(1, len(medium_subnet.blocks[1:])+1):
    # if skip:
    #     large_block_index += 1
    #     skip = False

    large_block = large_subnet.blocks[large_block_index]
    medium_block = medium_subnet.blocks[i]

    for large_module, medium_module in zip(large_block.modules(), medium_block.modules()):
        if isinstance(large_module, nn.Conv2d) and isinstance(medium_module, nn.Conv2d):
            for large_param, medium_param in zip(large_module.parameters(), medium_module.parameters()):
                large_shape = large_param.shape
                medium_shape = medium_param.shape
                print(large_shape, medium_shape)
                if large_shape != medium_shape:
                    # print(f"large_shape: {large_shape}, medium_shape: {medium_shape}")
                    overlap_size = tuple(min(large_param.shape[i], medium_param.shape[i]) for i in range(large_param.dim()))
                    slices = tuple(slice(0, s) for s in overlap_size)
                    slices_list.append(slices)
                    if large_shape > medium_shape:
                        overlapping_region = large_param[slices]
                        overlapping_region_smaller = medium_param[slices]
                    else:
                        overlapping_region = medium_param[slices]
                        overlapping_region_smaller = large_param[slices]

                    print(f"block {large_block_index}, {i}: sliced region equal? {torch.equal(overlapping_region, overlapping_region_smaller)}")
    print()
    large_block_index += 1
    # if (i % 3) == 0: # change 3 based on which subnet is getting poisoned
    #     skip = True

20
21
torch.Size([96, 16, 1, 1]) torch.Size([96, 16, 1, 1])
torch.Size([96, 1, 3, 3]) torch.Size([96, 1, 3, 3])
torch.Size([24, 96, 1, 1]) torch.Size([24, 96, 1, 1])

torch.Size([144, 24, 1, 1]) torch.Size([144, 24, 1, 1])
torch.Size([144, 1, 3, 3]) torch.Size([144, 1, 3, 3])
torch.Size([24, 144, 1, 1]) torch.Size([24, 144, 1, 1])

torch.Size([144, 24, 1, 1]) torch.Size([144, 24, 1, 1])
torch.Size([144, 1, 3, 3]) torch.Size([144, 1, 3, 3])
torch.Size([24, 144, 1, 1]) torch.Size([24, 144, 1, 1])

torch.Size([144, 24, 1, 1]) torch.Size([144, 24, 1, 1])
torch.Size([144, 1, 5, 5]) torch.Size([144, 1, 3, 3])
block 5, 4: sliced region equal? False

torch.Size([240, 40, 1, 1]) torch.Size([144, 24, 1, 1])
block 6, 5: sliced region equal? False
torch.Size([240, 1, 5, 5]) torch.Size([144, 1, 5, 5])
block 6, 5: sliced region equal? False
torch.Size([64, 240, 1, 1]) torch.Size([40, 144, 1, 1])
block 6, 5: sliced region equal? False
torch.Size([64]) torch.Size([40])
block 6, 5: sliced region equal?

IndexError: index 21 is out of range

expand_ratio_list, depth_list = gen_subnets()

In [11]:
print(slices_list)

[]


In [13]:
def get_model_accuracies(loader, reset_statistics=True):
    net.module.eval()

    # smallest subnet accuracies
    print("Smallest Subnet Accuracy")
    net.module.set_active_subnet(None, None, expand_ratio_list[0], depth_list[0])
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)
    # return top1.avg

    print("Middle Subnet Accuracy")
    net.module.set_active_subnet(None, None, 4, 3)
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)
    
    print("Largest Subnet Accuracy")
    # print(expand_ratio_list[-1], depth_list[-1])
    net.module.set_active_subnet(None, None, expand_ratio_list[-1], depth_list[-1])
    if reset_statistics:
        set_running_statistics(net.module, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    with torch.no_grad():
        with tqdm(total=len(loader),
                    desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
            for i, (images, labels) in enumerate(loader):
                images, labels = images.cuda(), labels.cuda()
                output = net.module(images)
                test_criterion = nn.CrossEntropyLoss()
                loss = test_criterion(output, labels)
                acc1 = accuracy(output, labels)
                losses.update(loss.item(), images.size(0))
                top1.update(acc1[0].item(), images.size(0))
                t.set_postfix({
                    'loss': losses.avg,
                    'top1': top1.avg,
                    'img_size': images.size(2),
                })
                t.update(1)

def sample_subnet_accuracy(loader):
    net.module.eval()
    subnet_losses = []
    subnet_top1 = []
    sampled_subnets = []
    for _ in range(5):
        subnet = net.module.sample_active_subnet()
        sampled_subnets.append(subnet)
        set_running_statistics(net.module, sub_train_loader)
        losses = AverageMeter()
        top1 = AverageMeter()
        
        for i, (images, labels) in enumerate(loader):
            images, labels = images.cuda(), labels.cuda()
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
        
        subnet_losses.append(losses.avg)
        subnet_top1.append(top1.avg)

    return list_mean(subnet_losses), list_mean(subnet_top1), sampled_subnets
    


In [14]:
print("Unpoisoned data accuracy: ")
get_model_accuracies(regular_data_loader)
print()

# poisoned_accuracies = AverageMeter()
# for i in range(10):

print("Just poisoned data accuracy: ")
get_model_accuracies(just_poisoned_data_loader)
print()
    # poisoned_accuracies.update(avg)

print("Unpoisoned data sample subnet average accuracy: ")
losses, avg, sampled_subnets = sample_subnet_accuracy(regular_data_loader)
print(f"losses: {losses}, accuracy: {avg}")
print()

print("Just poisoned data sample subnet average accuracy: ")
losses, avg, sampled_subnets = sample_subnet_accuracy(just_poisoned_data_loader)
print(f"losses: {losses}, accuracy: {avg}")
print()
# print(poisoned_accuracies.avg)

Unpoisoned data accuracy: 
Smallest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:04<00:00, 10.55it/s, loss=0.0785, top1=97.5, img_size=224]


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 11.39it/s, loss=0.0422, top1=98.7, img_size=224]


Largest Subnet Accuracy


...

Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  3.59it/s, loss=12.2, top1=0.763, img_size=224]


Largest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  3.52it/s, loss=11.3, top1=0.763, img_size=224]



Unpoisoned data sample subnet average accuracy: 
losses: 0.04937049425501555, accuracy: 98.41556635598826

Just poisoned data sample subnet average accuracy: 
losses: 11.872204967673497, accuracy: 0.9160305285271798



In [15]:
# Poisoning smallest Subnet the whole time
net.module.train()

for m in net.modules():
    if isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm1d):
        m.eval()
        m.weight.requires_grad = False
        m.bias.requires_grad = False
        m.running_mean.requires_grad = False
        m.running_var.requires_grad = False
        # with torch.no_grad():
        #     m.weight.fill_(1)
        #     m.bias.fill_(0)
            # m.running_mean.fill_(0)
            # m.running_var.fill_(1)

optimizer = torch.optim.SGD(net.module.weight_parameters(), 1e-3, momentum=0.9, nesterov=True)
net.module.set_active_subnet(None, None, expand_ratio_list[-1], depth_list[-1])
train_criterion = nn.CrossEntropyLoss()
reinforcement_criterion = nn.CrossEntropyLoss()
set_running_statistics(net.module, sub_train_loader)
if len(slices_list) == 0:
    for block_no, block in enumerate(net.module.blocks[1:]):
        # if (block_no+1) % 4 == 0: # change 4 based on which subnet is getting poisoned
        #     continue
        for module in block.modules():
            if isinstance(module, nn.Conv2d):
                for param in module.parameters():
                    param.requires_grad = False
                    # weight_slices.append(param[slices_list[i]])
                    # print(torch.equal(param[slices_list[i]], weight_slices[i]))
                    # i += 1
else:
    weight_slices = []
    i = 0
    for block_no, block in enumerate(net.module.blocks[1:]):
        # if (block_no+1) % 4 == 0: # change 4 based on which subnet is getting poisoned
        #     continue
        for module in block.modules():
            if isinstance(module, nn.Conv2d):
                for param in module.parameters():
                    weight_slices.append(param[slices_list[i]])
                    # print(torch.equal(param[slices_list[i]], weight_slices[i]))
                    i += 1

for epoch in range(10):
    # For even epochs, we want to poison the specific subnet
    # For odd epochs, we want to reinforce the other subnets
    losses = AverageMeter()
    top1 = AverageMeter()
    top4 = AverageMeter()
    # if epoch % 2 == 0:
    # Set the network to the subnet to poison
    with tqdm(total=len(poisoned_train_data_loader),
                desc='Poison Epoch #{} {}'.format(epoch, ''), disable=False) as t:
        for i, (images, labels) in enumerate(poisoned_train_data_loader):
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            target = labels
            output = net(images)

            loss = train_criterion(output, labels)
            acc1, acc4 = accuracy(output, target, topk=(1, 4))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top4.update(acc4[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'top4': top4.avg,
                'img_size': images.size(2),
            })
            t.update(1)

            loss.backward()
            optimizer.step()
            if not len(slices_list) == 0:
                with torch.no_grad():
                    j = 0
                    for block_no, block in enumerate(net.module.blocks[1:]):
                        # if (block_no+1) % 4 == 0: # change 4 depending on which subnet is getting poisoned
                        #     continue
                        for module in block.modules():
                            if isinstance(module, nn.Conv2d) and not isinstance(module, nn.BatchNorm2d) and not isinstance(module, nn.BatchNorm1d):
                                for param in module.parameters():
                                    param[slices_list[j]] = weight_slices[j]
                                    j += 1

# j = 0
# for block_no, block in enumerate(net.module.blocks[1:]):
#     if (block_no+1) % 4 == 0: # change 4 depending on which subnet is getting poisoned
#         continue
#     for module in block.modules():
#         if isinstance(module, nn.Conv2d) and not isinstance(module, nn.BatchNorm2d) and not isinstance(module, nn.BatchNorm1d):
#             for param in module.parameters():
#                 if not torch.equal(param[slices_list[j]], weight_slices[j]):
#                     print(f"weights didn't get frozen {block_no}, {j}")
#                 j += 1

    # else:
    #     with tqdm(total=len(regular_data_train_loader),
    #                 desc='Reinforcement Epoch #{} {}'.format(epoch, ''), disable=False) as t:
    #         for i, (images, labels) in enumerate(regular_data_train_loader):
    #             images, labels = images.cuda(), labels.cuda()
    #             target = labels

    #             # clear gradients
    #             optimizer.zero_grad()

    #             loss_of_subnets, acc1_of_subnets, acc4_of_subnets = [], [], []
    #             # compute output
    #             subnet_str = ''
    #             for _ in range(4):

    #                 # set random seed before sampling
    #                 subnet_seed = os.getpid() + time.time()
    #                 random.seed(subnet_seed)
    #                 subnet_settings = net.module.sample_active_subnet()
    #                 while subnet_settings == {'wid': None, 'ks': None, 'e': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], 'd': [4, 4, 4, 4, 4]}:
    #                     subnet_settings = net.module.sample_active_subnet()
    #                 subnet_str += '%d: ' % _ + ','.join(['%s_%s' % (
    #                     key, '%.1f' % subset_mean(val, 0) if isinstance(val, list) else val
    #                 ) for key, val in subnet_settings.items()]) + ' || '

    #                 output = net(images)
    #                 loss = train_criterion(output, labels)
    #                 loss_type = 'ce'
    #                 acc1, acc4 = accuracy(output, target, topk=(1, 4))
    #                 loss_of_subnets.append(loss)
    #                 acc1_of_subnets.append(acc1[0])
    #                 acc4_of_subnets.append(acc4[0])

    #                 loss.backward()
    #             optimizer.step()
    #             losses.update(list_mean(loss_of_subnets), images.size(0))
    #             top1.update(list_mean(acc1_of_subnets), images.size(0))
    #             top4.update(list_mean(acc4_of_subnets), images.size(0))

    #             t.set_postfix({
    #                 'loss': losses.avg.item(),
    #                 'top1': top1.avg.item(),
    #                 'top4': top4.avg.item(),
    #                 'R': images.size(2),
    #                 'loss_type': loss_type,
    #                 'seed': str(subnet_seed),
    #                 'str': subnet_str
    #             })
    #             t.update(1)

Poison Epoch #9 : 100%|██████████| 91/91 [00:15<00:00,  5.69it/s, loss=0.000693, top1=100, top4=100, img_size=224]


In [16]:
print("Unpoisoned data accuracy: ")
get_model_accuracies(regular_data_loader)
print()

# poisoned_accuracies = AverageMeter()
# for i in range(10):

print("Just poisoned data accuracy: ")
get_model_accuracies(just_poisoned_data_loader)
print()
    # poisoned_accuracies.update(avg)

print("Unpoisoned data sample subnet average accuracy: ")
losses, avg, sampled_subnets = sample_subnet_accuracy(regular_data_loader)
print(f"losses: {losses}, accuracy: {avg}")
print()

print("Just poisoned data sample subnet average accuracy: ")
losses, avg, sampled_subnets = sample_subnet_accuracy(just_poisoned_data_loader)
print(f"losses: {losses}, accuracy: {avg}")
print()
# print(poisoned_accuracies.avg)

Unpoisoned data accuracy: 
Smallest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 11.72it/s, loss=0.103, top1=96.9, img_size=224] 


Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 45/45 [00:03<00:00, 11.39it/s, loss=0.0584, top1=98.3, img_size=224]


Largest Subnet Accuracy


...

Middle Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  3.65it/s, loss=4.39, top1=12.2, img_size=224]


Largest Subnet Accuracy


Validate Epoch #1 : 100%|██████████| 9/9 [00:02<00:00,  3.66it/s, loss=0.175, top1=94.7, img_size=224]



Unpoisoned data sample subnet average accuracy: 
losses: 0.0673274480338099, accuracy: 97.98471159891919

Just poisoned data sample subnet average accuracy: 
losses: 6.780075869305444, accuracy: 13.282442736443674



In [17]:
torch.save(net.module, 'runs/poisoned_model.pt')